## What You'll Need

- A working `odsa-ai-backend/` project (cloned and `uv sync` completed)
- Your Anthropic API key in a `.env` file at the project root
- This notebook open in Jupyter (via VS Code, JupyterLab, etc.)

---

## 1. Setup & Environment

Before we can talk to Claude from Python, we need two things:
1. Our project dependencies installed (`uv sync` -- already done)
2. Our API key loaded from `.env`

The cell below sets up the Python path so we can import from `src/` and loads environment variables from your `.env` file.

In [ ]:
import sys
from pathlib import Path

# Find repo root by walking up until we find pyproject.toml
_here = Path(".").resolve()
_root = _here
while _root != _root.parent:
    if (_root / "pyproject.toml").exists():
        break
    _root = _root.parent
sys.path.insert(0, str(_root))

from dotenv import load_dotenv
load_dotenv(_root / ".env")

Let's verify your API key is loaded. The cell below checks that the `ANTHROPIC_API_KEY` environment variable exists. You should see a confirmation message -- **not** the key itself.

In [2]:
import os

key = os.getenv("ANTHROPIC_API_KEY")
if key and key.startswith("sk-ant-"):
    print(f"API key loaded successfully (starts with {key[:10]}...)")
elif key:
    print(f"WARNING: Key found but doesn't start with 'sk-ant-'. Check your key format.")
else:
    print("ERROR: No ANTHROPIC_API_KEY found. Check your .env file.")

API key loaded successfully (starts with sk-ant-api...)


Now import the pre-built module. This is `src/s0_generation/generate.py` -- it is **provided complete** as course infrastructure. You will use it throughout this notebook and future sessions.

In [3]:
from src.s0_generation.generate import call_claude, call_claude_with_usage

print("Imports successful -- ready to talk to Claude.")

Imports successful -- ready to talk to Claude.


---

## 2. Your First API Call

In AI-1, you typed prompts into a chat window. Now your Python code sends prompts to Claude.

Same concepts. New medium. Run the cell below.

In [4]:
# Your first API call -- Python talking to Claude
result = call_claude("Write me a one-sentence welcome message for a new AI engineer.")
print(result)

Welcome aboard! We're excited to have you join our team and can't wait to see the innovative AI solutions you'll help us build.


That response came from Claude's API -- not a browser, not copy-paste. Your Python code just had a conversation with Claude.

This is the foundation of everything we build in this course.

**Checkpoint 2:** You should see a response displayed above. If you don't, flag the instructor.

### Try it yourself

Change the prompt below to anything you want. Ask Claude a question, give it a task, have fun with it.

In [5]:
# Your turn -- change this prompt to anything you like
my_result = call_claude("What is one thing every Python developer should know?")
print(my_result)

One essential thing every Python developer should know is **how Python's object model and variable assignment work** — specifically that **variables are references to objects, not containers that hold values**.

This means:

```python
# Lists are mutable objects
a = [1, 2, 3]
b = a  # b references the SAME object as a
b.append(4)
print(a)  # [1, 2, 3, 4] - a changed too!

# vs immutable objects
x = 5
y = x  # y references the same int object
y = 10  # y now references a DIFFERENT object
print(x)  # 5 - x unchanged
```

This concept affects:
- **Function arguments** (mutations inside functions affect the original)
- **Default mutable arguments** (the classic `def func(lst=[]):` gotcha)
- **Copying** (shallow vs deep)
- **Performance understanding** (why string concatenation in loops is slow)

Understanding this prevents countless bugs and helps you write more Pythonic, efficient code. It's the foundation for grasping how Python really works under the hood.


---

## 3. Anatomy of a Response

When you call `client.messages.create()`, the SDK sends an HTTP POST request to Anthropic's servers. Claude processes your prompt and sends back a structured JSON response. The SDK parses that into a Python object.

```
Your Code                    Anthropic Servers
   |                              |
   |-- HTTP POST --------------->|
   |   headers: API key           |
   |   body: model, messages,     |
   |         max_tokens           |
   |                              |-- Claude processes
   |                              |
   |<------------ JSON response --|
   |   content, usage, model,     |
   |   stop_reason                |
```

The `call_claude()` function returns just the text. But there is a lot more in the response object. Let's look under the hood.

### Seeing the raw response object

To see everything the API returns, we can make a call directly with the `anthropic` SDK. This is what `call_claude()` does internally.

In [ ]:
import anthropic
import json

client = anthropic.Anthropic()

# Make a raw API call so we can inspect the full response object
message = client.messages.create(
    model="claude-sonnet-4-5",
    max_tokens=1024,
    system="You are a helpful assistant.",
    messages=[{"role": "user", "content": "What is an API in one sentence?"}]
)

# The full response object, pretty-printed as JSON
# The SDK returns a Pydantic model -- .model_dump() converts it to a plain dict
print("=== Full Response Object ===")
print(json.dumps(message.model_dump(), indent=2, default=str))
print()

That is a lot of information. Let's pull it apart piece by piece.

In [ ]:
# The actual text Claude generated
print("Text:", message.content[0].text)
print()

# Which model responded
print("Model:", message.model)
print()

# Why Claude stopped generating
# "end_turn" means Claude finished naturally
# "max_tokens" means Claude was cut off mid-sentence
print("Stop reason:", message.stop_reason)
print()

# The role of the response (always "assistant")
print("Role:", message.role)
print()

# Token usage -- how many tokens went in and came out
print("Input tokens:", message.usage.input_tokens)
print("Output tokens:", message.usage.output_tokens)

### Why `message.content[0].text`?

The API can return multiple content blocks (for example, text and images). For text-only responses, there is always exactly one block at index `[0]`. We always grab the first one.

### The Messages List -- Chat Window to API Call

In AI-1, you had conversations in a chat window. Every back-and-forth exchange maps directly to the `messages` list:

```
Chat Window (AI-1)              messages=[
                                  {"role": "user", "content": "Hi"},
You: Hi               -->        {"role": "assistant", "content": "Hello!"},
Claude: Hello!         -->        {"role": "user", "content": "What is AI?"},
You: What is AI?       -->      ]
```

The `system` prompt is a separate parameter -- not part of the messages list. Everything you learned about prompt structure in AI-1 applies here.

In [ ]:
# A multi-turn conversation as a messages list
# This is what a chat history looks like in code
messages = [
    {"role": "user", "content": "What is Northbrook Partners?"},
    {"role": "assistant", "content": "I don't have specific information about Northbrook Partners. Could you tell me more about what you'd like to know?"},
    {"role": "user", "content": "It's a fictional financial advisory firm we'll be using as our example throughout this course. What kinds of documents might a firm like that have?"}
]

response = client.messages.create(
    model="claude-sonnet-4-5",
    max_tokens=1024,
    messages=messages
)

print(response.content[0].text)

Notice how Claude's response takes the full conversation history into account. The `messages` list is how multi-turn conversations work programmatically. For today, single-turn calls are enough -- we will use multi-turn patterns later in the course.

---

## 4. Token Tracking

Remember tokens from AI-1? Now you can see exactly how many you used -- and what they cost.

The `call_claude_with_usage()` function returns a dictionary with the response text **plus** metadata about the call.

In [ ]:
# call_claude_with_usage returns a dict with text + metadata
result = call_claude_with_usage("Explain what an API key is in two sentences.")

# Print every key-value pair in the result
for key, value in result.items():
    print(f"{key}: {value}")

**Checkpoint 4:** You should see the response text plus `input_tokens`, `output_tokens`, `model`, and `stop_reason` printed above.

### Estimating cost

API pricing is per token. For Sonnet 4.5 (approximate):
- **Input:** $3 per 1 million tokens
- **Output:** $15 per 1 million tokens

Let's calculate the cost of that call.

In [ ]:
# Sonnet 4.5 pricing (per token)
INPUT_COST_PER_TOKEN = 3.00 / 1_000_000   # $3 per 1M input tokens
OUTPUT_COST_PER_TOKEN = 15.00 / 1_000_000  # $15 per 1M output tokens

input_cost = result["input_tokens"] * INPUT_COST_PER_TOKEN
output_cost = result["output_tokens"] * OUTPUT_COST_PER_TOKEN
total_cost = input_cost + output_cost

print(f"Input tokens:  {result['input_tokens']:>6}  (${input_cost:.6f})")
print(f"Output tokens: {result['output_tokens']:>6}  (${output_cost:.6f})")
print(f"Total cost:             (${total_cost:.6f})")
print()
print(f"At this rate, 1,000 identical calls would cost: ${total_cost * 1000:.2f}")

A single short call costs fractions of a cent. But when you run 1,000 of these in a batch pipeline, the math starts to matter. Tracking tokens from day one is a professional habit.

---

## 5. System Prompts

The `system` parameter is the same system prompt you used in AI-1 -- but now it is a parameter you pass programmatically. Same question, different personas, different results.

In [ ]:
question = "Why is Python popular for data work?"

# Default system prompt
response_default = call_claude(question)
print("=== Default ===")
print(response_default)
print()

In [ ]:
# Concise technical expert
response_expert = call_claude(
    question,
    system_prompt="You are a senior data engineer. Be concise -- three bullet points maximum."
)
print("=== Technical Expert ===")
print(response_expert)
print()

In [ ]:
# Enthusiastic teacher
response_teacher = call_claude(
    question,
    system_prompt="You are an enthusiastic programming teacher explaining to a complete beginner. Use analogies."
)
print("=== Enthusiastic Teacher ===")
print(response_teacher)
print()

Same question, three very different outputs. The system prompt controls **how** Claude responds -- tone, length, format, audience level. In AI-1 you learned this in the chat window. Now you set it in code.

### Try it yourself

Change the system prompt below. Try a pirate, a Shakespearean actor, a haiku poet, a corporate lawyer -- whatever you want. Re-run the cell and see what happens.

In [ ]:
# Your turn -- experiment with system prompts
my_response = call_claude(
    "Explain what an API is.",
    system_prompt="You are a pirate. Respond in pirate speak."
)
print(my_response)

---

## 6. Temperature Exploration

Temperature controls randomness.
- **0.0** = Deterministic (same input produces nearly the same output every time)
- **1.0** = Creative (same input produces varied output each time)

For pipeline work (extraction, classification, factual answers), you usually want **low temperature**. Predictability matters.

Let's see the difference.

In [28]:
prompt = "Write a one-sentence opening line for a mystery novel."

# Run the same prompt 3 times at temperature 0.0
print("=== Temperature 0.0 (deterministic) ===")
for i in range(3):
    result = call_claude(prompt, temperature=0.0)
    print(f"  Run {i+1}: {result}")
print()

=== Temperature 0.0 (deterministic) ===
  Run 1: The detective found the mayor's body in the library at midnight, but the coroner would later confirm he'd been dead for three days.
  Run 2: The detective found the mayor's body in the library at midnight, but the coroner would later confirm he'd been dead for three days.
  Run 3: The detective found the mayor's body in the library at midnight, but the coroner would later confirm he'd been dead for three days.



In [29]:
# Run the same prompt 3 times at temperature 1.0
print("=== Temperature 1.0 (creative) ===")
for i in range(3):
    result = call_claude(prompt, temperature=1.0)
    print(f"  Run {i+1}: {result}")
print()

=== Temperature 1.0 (creative) ===
  Run 1: The detective found the mayor's body at dawn, but the real mystery was why the victim had been smiling.
  Run 2: The morning fog lifted to reveal Detective Sarah Chen standing over a body that had been dead for three days—in a room locked from the inside.
  Run 3: The detective found the mayor's body in the library at midnight, but according to the coroner, he'd been dead for three days.



At temperature 0.0, you should see very similar (or identical) responses each run. At temperature 1.0, you should see variation. This is a key engineering decision you will make throughout this course: do you want consistency or creativity?

### Try it yourself

Pick a creative prompt and run it at both temperatures. See how much the output varies.

In [27]:
# Your turn -- experiment with temperature
my_prompt = "Write a one-sentence tagline for a coffee shop."

print("Temperature 0.0:")
for i in range(3):
    print(f"  {call_claude(my_prompt, temperature=0.0)}")

print("\nTemperature 1.0:")
for i in range(3):
    print(f"  {call_claude(my_prompt, temperature=1.0)}")

Temperature 0.0:
  "Where every cup tells a story and every sip feels like home."
  "Where every cup tells a story and every sip feels like home."
  "Where every cup tells a story and every sip feels like home."

Temperature 1.0:
  "Where every cup tells a story and every moment feels like home."
  "Where every cup tells a story and every sip feels like home."
  "Where every cup tells a story, and every sip feels like home."


---

## 7. Max Tokens

`max_tokens` is a ceiling, not a target. If Claude's natural answer is 50 tokens and you set `max_tokens=2000`, you get 50 tokens. But if the natural answer is 500 tokens and you set `max_tokens=100`, Claude gets **cut off mid-sentence**.

The way to detect truncation: check `stop_reason`.
- `"end_turn"` = Claude finished naturally
- `"max_tokens"` = Claude was cut off

In [ ]:
# Plenty of room -- Claude finishes naturally
result_full = call_claude_with_usage(
    "List the top 5 programming languages and one reason each is popular.",
    max_tokens=1024
)
print("=== max_tokens=1024 ===")
print(f"Stop reason: {result_full['stop_reason']}")
print(f"Output tokens used: {result_full['output_tokens']}")
print(f"Response:\n{result_full['text']}")
print()

In [ ]:
# Too tight -- Claude gets cut off
result_truncated = call_claude_with_usage(
    "List the top 5 programming languages and one reason each is popular.",
    max_tokens=50
)
print("=== max_tokens=50 ===")
print(f"Stop reason: {result_truncated['stop_reason']}")
print(f"Output tokens used: {result_truncated['output_tokens']}")
print(f"Response:\n{result_truncated['text']}")
print()
print("Notice: stop_reason changed. Claude was mid-sentence when the limit hit.")

### Try it yourself

Run the same prompt with `max_tokens=100`, `max_tokens=500`, and `max_tokens=2000`. Watch how the output and `stop_reason` change.

In [ ]:
# Your turn -- experiment with max_tokens
prompt = "List the top 10 programming languages and explain why each is popular."

for token_limit in [100, 500, 2000]:
    result = call_claude_with_usage(prompt, max_tokens=token_limit)
    print(f"=== max_tokens={token_limit} ===")
    print(f"Stop reason: {result['stop_reason']}")
    print(f"Output tokens: {result['output_tokens']}")
    # Show just the first 200 characters to keep output manageable
    preview = result['text'][:200]
    print(f"Preview: {preview}...")
    print()

---

## 8. Comparing Models

Different models have different strengths. Let's compare Haiku (fast, cheap) and Sonnet (balanced, our default) on the same prompt.

In [ ]:
comparison_prompt = "Explain gravity in two sentences."

# Haiku -- fast and cheap
result_haiku = call_claude_with_usage(comparison_prompt, model="claude-haiku-4-5")

# Sonnet -- balanced (our course default)
result_sonnet = call_claude_with_usage(comparison_prompt, model="claude-sonnet-4-5")

print("=== Haiku 4.5 ===")
print(f"Response: {result_haiku['text']}")
print(f"Input tokens: {result_haiku['input_tokens']} | Output tokens: {result_haiku['output_tokens']}")
print()

print("=== Sonnet 4.5 ===")
print(f"Response: {result_sonnet['text']}")
print(f"Input tokens: {result_sonnet['input_tokens']} | Output tokens: {result_sonnet['output_tokens']}")
print()

print("Choosing the right model for the right task is a real engineering decision.")
print("We'll explore this more throughout the certificate.")

**Checkpoint 5:** You should have run calls with at least two different model or parameter configurations.

---

## 9. The Pre-Built Module -- What's Under the Hood

You have been using `call_claude()` and `call_claude_with_usage()` all session. These functions live in `src/s0_generation/generate.py` -- a pre-built module that is part of your course infrastructure.

Here is what `call_claude()` looks like. You do not need to modify this file -- but you should understand it.

```python
def call_claude(
    prompt: str,
    system_prompt: str = "You are a helpful assistant.",
    model: str = "claude-sonnet-4-5",
    max_tokens: int = 1024,
    temperature: float = 1.0,
) -> str:
    """Send a prompt to Claude and return the response text."""
    client = anthropic.Anthropic()
    response = client.messages.create(
        model=model,
        max_tokens=max_tokens,
        temperature=temperature,
        system=system_prompt,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.content[0].text
```

Key design decisions:
- **Sensible defaults** -- call it with just a prompt, or customize everything
- **Returns just the text** -- simple for most use cases
- The `call_claude_with_usage()` variant returns a dictionary with metadata (tokens, model, stop reason) for when you need to track costs and debug

This is the pattern for AI-2: pre-built `.py` modules provide working infrastructure. You explore and use them from notebooks.

---

## 10. Parameter Experiments

Independent exploration time. Pick one or more of the challenges below and experiment. These cells are yours to modify, re-run, and learn from.

### Challenge 1: The System Prompt Experiment

Write three different system prompts for the same question. Which produces the most useful output?

In [ ]:
# Challenge 1: System Prompt Experiment
question = "What makes a good data pipeline?"

prompts = [
    "You are a helpful assistant.",
    "You are a senior data engineer. Be concise and practical.",
    "You are a teacher explaining to someone who has never written code. Use analogies."
]

for i, sp in enumerate(prompts, 1):
    result = call_claude(question, system_prompt=sp)
    print(f"=== System Prompt {i}: {sp[:50]}... ===")
    print(result)
    print()

### Challenge 2: The Token Budget Experiment

Ask Claude to list the top 10 programming languages. Run with `max_tokens=100`, then `500`, then `2000`. What changes?

In [ ]:
# Challenge 2: Token Budget Experiment
prompt = "List the top 10 programming languages and explain why each is popular."

for limit in [100, 500, 2000]:
    result = call_claude_with_usage(prompt, max_tokens=limit)
    print(f"=== max_tokens={limit} ===")
    print(f"Stop reason: {result['stop_reason']} | Output tokens: {result['output_tokens']}")
    print(result['text'][:300])  # Show first 300 chars
    print("..." if len(result['text']) > 300 else "")
    print()

### Challenge 3: The Temperature Experiment

Pick any prompt. Run it 3 times at `temperature=0.0` and 3 times at `temperature=1.0`. How consistent are the results?

In [ ]:
# Challenge 3: Temperature Experiment
prompt = "Give me a creative name for a Python package that does data validation."

print("=== Temperature 0.0 ===")
for i in range(3):
    print(f"  Run {i+1}: {call_claude(prompt, temperature=0.0)}")

print("\n=== Temperature 1.0 ===")
for i in range(3):
    print(f"  Run {i+1}: {call_claude(prompt, temperature=1.0)}")

### Bonus: Your own experiment

Use this cell for anything you want to try. Change prompts, combine parameters, run comparisons.

In [ ]:
# Your own experiment -- go wild


---

## 11. The Messages Bridge -- What's Next

Today you sent prompts and got back text. But what if, instead of asking for prose, you asked for **JSON**?

```python
result = call_claude(
    "Extract the company name and founding year from this text: "
    "'Northbrook Partners was established in 2018 as a boutique financial advisory firm.'",
    system_prompt="Respond with valid JSON only. Format: {\"company\": ..., \"year\": ...}"
)
```

This is the bridge to Session 1.2. Next time, we will:
- Process **batches** of documents through Claude
- Force Claude's output into **specific shapes** using Pydantic schemas
- Build **error handling** so one bad document never crashes the pipeline

The `call_claude()` function you used today becomes the engine that powers the extraction pipeline we build in the Northbrook Partners corpus.

In [ ]:
# A preview of what's coming -- extracting structured data
result = call_claude(
    "Extract the company name and founding year from this text: "
    "'Northbrook Partners was established in 2018 as a boutique financial advisory firm.'",
    system_prompt="Respond with valid JSON only. Format: {\"company\": ..., \"year\": ...}",
    temperature=0.0
)
print("Claude returned:")
print(result)
print()
print("But how do we GUARANTEE this is valid JSON? How do we validate the fields?")
print("That's Session 1.2.")

---

## 12. Session Recap

### What you accomplished today:

- Set up a working Python project with `uv` and API key management
- Made your first API call to Claude from Python
- Explored the full response object (content, usage, model, stop_reason)
- Tracked token usage and estimated costs
- Experimented with system prompts, temperature, max_tokens, and model selection
- Previewed structured data extraction (the bridge to Session 1.2)

### Key takeaways:

1. **The chat window is an API call.** Everything from AI-1 maps directly to code parameters.
2. **Track metadata from day one.** Token counts, stop reasons, and model choices are engineering decisions, not afterthoughts.
3. **Sensible defaults, optional overrides.** The `call_claude()` function works with just a prompt, but lets you control everything.

### Questions to sit with before Session 1.2:

> **On Structure:** "When you get data from an API, how do you know it's in the format you expected? What happens when it's not?"

> **On Scale:** "If you had to process 1,000 documents through Claude, what would break first?"

> **On Validation:** "How do you test that your code handles edge cases before they hit production?"

### Homework (not graded):

**The Parameter Report** -- Pick a task you do at work. Write a prompt for it. Run with 3+ configurations (different models, temperatures, system prompts). Record token usage, assess quality, and write 3-5 sentences about which configuration was best and why. Save as `experiments/session_1_1_report.txt`.